# Deep Learning for Biological Modeling
### SIAM Life Sciences 2026 Mini-Tutorial — Deep Learning Section

This notebook walks through three examples:
1. **MNIST 3 vs 8 — MLP**: A simple neural network baseline
2. **MNIST 3 vs 8 — CNN**: Better performance + interpretability with Grad-CAM
3. **PneumoniaMNIST — CNN**: Same approach on real biomedical images

**Run in Google Colab.** Execute the setup cell first, then run cells top to bottom.


## Setup

In [1]:
# Install any extra packages (medmnist is not in Colab by default)
!pip install medmnist -q


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.ndimage as ndi
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
import medmnist
from medmnist import INFO

print("TensorFlow version:", tf.__version__)


---
## Part 1 — MNIST (3 vs. 8): A Simple Neural Network (MLP)

We start with the classic MNIST handwritten digit dataset, but restrict
it to just two classes: **3** and **8**.  These two digits are visually
similar, making the task non-trivial.

**Goal**: train a *Multi-Layer Perceptron* (MLP) — a fully-connected
network — and see how well it does.


### 1.1 Load and preview the data

In [ ]:
# Load full MNIST
(x_train_full, y_train_full), (x_test_full, y_test_full) = keras.datasets.mnist.load_data()

# Keep only 3s and 8s, re-label them 0 and 1
def filter_mnist(x, y, keep=(3, 8)):
    mask = (y == keep[0]) | (y == keep[1])
    x_f = x[mask].astype("float32") / 255.0   # normalise to [0, 1]
    y_f = (y[mask] == keep[1]).astype("float32")  # 3 → 0,  8 → 1
    return x_f, y_f

x_train, y_train = filter_mnist(x_train_full, y_train_full)
x_test,  y_test  = filter_mnist(x_test_full,  y_test_full)

print(f"Training samples: {len(x_train)}  |  Test samples: {len(x_test)}")
print(f"Image shape: {x_train.shape[1:]}")


In [ ]:
# Show a handful of examples
fig, axes = plt.subplots(2, 5, figsize=(10, 4))
label_map = {0: "Digit 3", 1: "Digit 8"}

for i, ax in enumerate(axes.flat):
    ax.imshow(x_train[i], cmap="gray")
    ax.set_title(label_map[int(y_train[i])], fontsize=9)
    ax.axis("off")
plt.suptitle("Sample training images", y=1.01)
plt.tight_layout()
plt.show()


### 1.2 Build and train the MLP

An MLP **flattens** each 28×28 image into a vector of 784 numbers and
passes it through one (or more) fully-connected layers.

We use just **one hidden layer** here to keep things transparent.
Adding more layers is easy — just copy the `Dense` line.


In [ ]:
def build_mlp():
    model = models.Sequential([
        layers.Flatten(input_shape=(28, 28)),      # 28x28 → 784
        layers.Dense(64, activation="relu"),        # 1 hidden layer
        layers.Dense(1,  activation="sigmoid"),     # binary output
    ], name="MLP")
    model.compile(optimizer="adam",
                  loss="binary_crossentropy",
                  metrics=["accuracy"])
    return model

mlp = build_mlp()
mlp.summary()


In [ ]:
mlp_history = mlp.fit(
    x_train, y_train,
    validation_data=(x_test, y_test),
    epochs=10,
    batch_size=64,
    verbose=1
)


In [ ]:
# Plot training vs validation accuracy
plt.figure(figsize=(7, 3.5))
plt.plot(mlp_history.history["accuracy"],     label="Train accuracy")
plt.plot(mlp_history.history["val_accuracy"], label="Val accuracy", linestyle="--")
plt.xlabel("Epoch"); plt.ylabel("Accuracy")
plt.title("MLP — Training curve")
plt.legend(); plt.tight_layout(); plt.show()

mlp_loss, mlp_acc = mlp.evaluate(x_test, y_test, verbose=0)
print(f"MLP test accuracy: {mlp_acc:.3f}")


---
## Part 2 — MNIST (3 vs. 8): CNN + Interpretability

A **Convolutional Neural Network (CNN)** treats the image as a 2-D grid
rather than a flat vector.  Convolutional filters slide over the image
and learn to detect local patterns (edges, curves, textures).

We'll see that the CNN usually outperforms the MLP on image tasks —
and then we'll look *inside* it using **Grad-CAM**.


### 2.1 Build and train the CNN

In [ ]:
def build_cnn(input_shape=(28, 28)):
    """Simple 2-layer CNN.  We'll reuse this same architecture for PneumoniaMNIST."""
    model = models.Sequential([
        layers.Reshape((28, 28, 1), input_shape=input_shape),  # add channel dim
        layers.Conv2D(16, (3, 3), activation="relu", padding="same", name="conv1"),
        layers.MaxPooling2D((2, 2)),
        layers.Conv2D(32, (3, 3), activation="relu", padding="same", name="conv2"),
        layers.MaxPooling2D((2, 2)),
        layers.Flatten(),
        layers.Dense(64, activation="relu"),
        layers.Dense(1,  activation="sigmoid"),
    ], name="CNN")
    model.compile(optimizer="adam",
                  loss="binary_crossentropy",
                  metrics=["accuracy"])
    return model

cnn_mnist = build_cnn()
cnn_mnist.summary()


In [ ]:
cnn_mnist_history = cnn_mnist.fit(
    x_train, y_train,
    validation_data=(x_test, y_test),
    epochs=10,
    batch_size=64,
    verbose=1
)


In [ ]:
# Compare MLP vs CNN
cnn_loss, cnn_acc = cnn_mnist.evaluate(x_test, y_test, verbose=0)

print(f"MLP test accuracy: {mlp_acc:.3f}")
print(f"CNN test accuracy: {cnn_acc:.3f}")

# Side-by-side accuracy curves
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, hist, title in zip(axes,
                            [mlp_history, cnn_mnist_history],
                            ["MLP", "CNN"]):
    ax.plot(hist.history["accuracy"],     label="Train")
    ax.plot(hist.history["val_accuracy"], label="Val", linestyle="--")
    ax.set_title(f"{title} accuracy"); ax.set_xlabel("Epoch")
    ax.set_ylabel("Accuracy"); ax.legend()
plt.tight_layout(); plt.show()


### 2.2 Grad-CAM: Where is the network looking?

**Grad-CAM** (Gradient-weighted Class Activation Mapping) answers the
question: *which spatial regions of the image most influenced the
network's prediction?*

**How it works (in brief):**
1. Pick the last convolutional layer's feature maps.
2. Compute the gradient of the predicted score with respect to those maps.
3. Average-pool the gradients to get a weight per feature map.
4. Take a weighted sum of the feature maps → a low-resolution heatmap.
5. Upsample and overlay on the original image.

A **smoothed** version applies a small Gaussian blur to reduce noise.


In [ ]:
def grad_cam(model, image, last_conv_layer="conv2"):
    """
    Compute a Grad-CAM heatmap for a single 28x28 grayscale image.
    Returns a heatmap the same size as the input (28x28).
    """
    # Build a sub-model that outputs (conv activations, final prediction)
    grad_model = tf.keras.Model(
        inputs=model.inputs,
        outputs=[model.get_layer(last_conv_layer).output, model.output]
    )

    img_tensor = tf.cast(image[np.newaxis, ...], tf.float32)  # (1, 28, 28)

    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(img_tensor)
        # For binary sigmoid: use the output directly as the "score"
        loss = predictions[0, 0]

    # Gradient of score w.r.t. conv feature maps
    grads = tape.gradient(loss, conv_outputs)            # (1, H, W, C)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2)) # (C,)

    # Weight each feature map by its pooled gradient
    conv_outputs = conv_outputs[0]                        # (H, W, C)
    heatmap = tf.reduce_sum(conv_outputs * pooled_grads, axis=-1).numpy()  # (H, W)

    # ReLU (keep only positive influence) + normalise to [0, 1]
    heatmap = np.maximum(heatmap, 0)
    if heatmap.max() > 0:
        heatmap /= heatmap.max()

    # Resize to original image size (28x28)
    zoom_factors = (image.shape[0] / heatmap.shape[0],
                    image.shape[1] / heatmap.shape[1])
    heatmap = ndi.zoom(heatmap, zoom_factors)
    return heatmap


def smooth_cam(heatmap, sigma=2):
    """Gaussian-smooth a heatmap to reduce noise."""
    return ndi.gaussian_filter(heatmap, sigma=sigma)


In [ ]:
def visualise_cam(model, images, labels, indices, label_map, title_prefix=""):
    """Plot original | Grad-CAM overlay | Smoothed Grad-CAM overlay for each index."""
    fig, axes = plt.subplots(len(indices), 3,
                             figsize=(9, 3.2 * len(indices)))
    if len(indices) == 1:
        axes = axes[np.newaxis, :]

    col_titles = ["Original", "Grad-CAM", "Smoothed Grad-CAM"]
    for ax, ct in zip(axes[0], col_titles):
        ax.set_title(ct, fontsize=10, fontweight="bold")

    for row, idx in enumerate(indices):
        img = images[idx]
        true_label = label_map[int(labels[idx])]

        cam = grad_cam(model, img)
        cam_smooth = smooth_cam(cam)

        # Column 0: original
        axes[row, 0].imshow(img, cmap="gray")
        axes[row, 0].set_ylabel(f"True: {true_label}", fontsize=9)

        # Columns 1 & 2: overlay
        for col, heatmap in enumerate([cam, cam_smooth], start=1):
            axes[row, col].imshow(img, cmap="gray")
            axes[row, col].imshow(heatmap, cmap="jet", alpha=0.45)

        for ax in axes[row]:
            ax.axis("off")

    plt.suptitle(f"{title_prefix}Grad-CAM", y=1.01)
    plt.tight_layout()
    plt.show()


# Pick one 3 and one 8 from the test set
idx_3 = np.where(y_test == 0)[0][5]
idx_8 = np.where(y_test == 1)[0][5]
label_map_mnist = {0: "Digit 3", 1: "Digit 8"}

visualise_cam(cnn_mnist, x_test, y_test,
              indices=[idx_3, idx_8],
              label_map=label_map_mnist,
              title_prefix="MNIST CNN — ")


**What to notice:**
- The raw Grad-CAM highlights the regions the CNN relies on most.
- The smoothed version makes the pattern easier to see.
- For digit **3**, does the heatmap focus on the curves?
- For digit **8**, does it activate near the loops?

This is the core idea we'll take into medical imaging next.


---
## Part 3 — PneumoniaMNIST: Interpretability in Medical Imaging

We now apply the **exact same CNN architecture and Grad-CAM workflow**
to a real biomedical dataset.

**PneumoniaMNIST** contains pediatric chest X-rays resized to 28×28,
labelled as *Normal* (0) or *Pneumonia* (1).

The question isn't just "can we classify?" but
**"does the network look at the right part of the chest?"**


### 3.1 Load and preview PneumoniaMNIST

In [ ]:
# Download PneumoniaMNIST via the medmnist package
DataClass = getattr(medmnist, INFO['pneumoniamnist']['python_class'])

train_data = DataClass(split='train', download=True)
test_data  = DataClass(split='test',  download=True)

def extract_arrays(dataset):
    imgs   = np.array([item[0] for item in dataset], dtype="float32") / 255.0
    labels = np.array([item[1][0] for item in dataset], dtype="float32")
    return imgs.squeeze(), labels   # squeeze: (N,28,28,1) → (N,28,28)

x_pneu_train, y_pneu_train = extract_arrays(train_data)
x_pneu_test,  y_pneu_test  = extract_arrays(test_data)

print(f"Training: {len(x_pneu_train)} samples  |  Test: {len(x_pneu_test)} samples")
print(f"Class balance (train) — Normal: {(y_pneu_train==0).sum()}  Pneumonia: {(y_pneu_train==1).sum()}")


In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(10, 4.5))
label_map_pneu = {0: "Normal", 1: "Pneumonia"}

idx_normal    = np.where(y_pneu_train == 0)[0][:5]
idx_pneumonia = np.where(y_pneu_train == 1)[0][:5]

for row, (indices, row_title) in enumerate([(idx_normal, "Normal"),
                                             (idx_pneumonia, "Pneumonia")]):
    for col, idx in enumerate(indices):
        ax = axes[row, col]
        ax.imshow(x_pneu_train[idx], cmap="gray")
        if col == 0:
            ax.set_ylabel(row_title, fontsize=10)
        ax.axis("off")

plt.suptitle("PneumoniaMNIST sample images")
plt.tight_layout(); plt.show()


### 3.2 Train the CNN (same architecture as before)

No changes to the network — we're deliberately reusing the same
`build_cnn()` function to show that one architecture can transfer
across different image classification problems.


In [ ]:
cnn_pneu = build_cnn()  # same architecture as MNIST CNN

cnn_pneu_history = cnn_pneu.fit(
    x_pneu_train, y_pneu_train,
    validation_data=(x_pneu_test, y_pneu_test),
    epochs=10,
    batch_size=64,
    verbose=1
)

pneu_loss, pneu_acc = cnn_pneu.evaluate(x_pneu_test, y_pneu_test, verbose=0)
print(f"\nPneumoniaMNIST CNN test accuracy: {pneu_acc:.3f}")


### 3.3 Smoothed Grad-CAM on Chest X-rays

Now we ask: is the network attending to the **lungs** when it predicts
pneumonia — or is it finding spurious patterns?

A clinician looking at the heatmap can immediately judge whether the
model is reasoning anatomically.


In [ ]:
# Pick one Normal and one Pneumonia X-ray from the test set
idx_norm = np.where(y_pneu_test == 0)[0][2]
idx_pneu = np.where(y_pneu_test == 1)[0][2]

visualise_cam(cnn_pneu, x_pneu_test, y_pneu_test,
              indices=[idx_norm, idx_pneu],
              label_map=label_map_pneu,
              title_prefix="PneumoniaMNIST CNN — ")


**Discussion questions:**
1. For a pneumonia case, does the heatmap concentrate on the lung fields?
   If it activates strongly over the heart shadow or ribs, what might that suggest?
2. Compare the Grad-CAM from the MLP (if you run it) vs. the CNN.
   Why does the CNN tend to produce more spatially coherent maps?
3. At 28×28 resolution, fine anatomical detail is lost.
   How might results differ on full-resolution X-rays?
4. In a clinical setting, what would it take to *trust* a model's
   saliency map enough to use it in diagnosis?


---
## Summary

| Model | Dataset | Test Accuracy |
|-------|---------|---------------|
| MLP (1 hidden layer) | MNIST 3 vs 8 | *(see above)* |
| CNN (2 conv layers)  | MNIST 3 vs 8 | *(see above)* |
| CNN (same)           | PneumoniaMNIST | *(see above)* |

**Key takeaways:**
- CNNs exploit spatial structure in images — typically outperforming MLPs on image tasks.
- Grad-CAM lets us probe *where* the network focuses, not just *how well* it performs.
- The same architecture and interpretability tools transfer across datasets.
- Interpretability is especially valuable in biomedical settings, where we need to audit model reasoning.

**Next up:** Deep-Symbolic Optimization for learning ODEs from data.
